# 🏯 10张测试 · 国风4 XL (GuoFeng4)
## 全自动，无需下载任何东西！
模型从 HuggingFace 自动拉取

In [ ]:
# @title 1. 安装依赖
!pip install diffusers transformers accelerate xformers pillow controlnet-aux huggingface_hub -q
import torch, os, json, time, gc
from pathlib import Path
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from controlnet_aux import CannyDetector
from huggingface_hub import hf_hub_download
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. 10张测试卡牌
cards = [
    {"card_id":"QH-P-0001-L05","name":"秦始皇","type":"person","dynasty":"秦汉","tags":["帝王","秦朝","统一","长城"],"story":"秦始皇嬴政，中国历史上第一位皇帝。"},
    {"card_id":"SG-P-0002-L04","name":"关羽","type":"person","dynasty":"三国","tags":["五虎将","忠义","武圣"],"story":"关羽，蜀汉五虎将之首，被后世尊为武圣关帝。"},
    {"card_id":"ST-P-0001-L05","name":"李世民","type":"person","dynasty":"隋唐","tags":["唐朝","帝王","贞观之治"],"story":"唐太宗李世民，开创贞观之治的一代明君。"},
    {"card_id":"QH-E-0017-L03","name":"鸿门宴","type":"event","dynasty":"秦汉","tags":["楚汉","宴会","项羽","刘邦"],"story":"项羽设宴鸿门欲杀刘邦。"},
    {"card_id":"SG-E-0018-L04","name":"赤壁之战","type":"event","dynasty":"三国","tags":["火攻","周瑜","曹操"],"story":"孙刘联军火烧曹船。"},
    {"card_id":"QH-L-0004-L01","name":"咸阳","type":"place","dynasty":"秦汉","tags":["秦都","关中","宫殿"],"story":"咸阳是秦朝都城，中国历史上第一座大一统帝国首都。"},
    {"card_id":"SG-L-0001-L01","name":"赤壁","type":"place","dynasty":"三国","tags":["长江","古战场"],"story":"赤壁之战火烧连船，三国鼎立格局从此奠定。"},
    {"card_id":"QH-W-0025-L03","name":"天子剑","type":"weapon","dynasty":"秦汉","tags":["帝王","宝剑"],"story":"天子剑象征皇权至高无上。"},
    {"card_id":"QH-B-0029-L03","name":"九章算术","type":"classic","dynasty":"秦汉","tags":["数学","汉代"],"story":"汉代成书的中国古代最重要的数学经典。"},
    {"card_id":"QH-D-0035-L05","name":"大秦帝国","type":"dynasty","dynasty":"秦汉","tags":["秦朝","帝国","统一"],"story":"大秦帝国是中国历史上第一个大一统王朝。"},
]
print(f"📋 {len(cards)} 张")

In [ ]:
# @title 3. 🔥 自动下载并加载国风4 XL（无需手动下载任何文件）
canny = CannyDetector()

print("⏳ 下载国风4 XL模型 (约6.5GB，仅首次需要，后续缓存)...")
model_path = hf_hub_download(
    repo_id="xiaolxl/GuoFeng4_XL",
    filename="GuoFeng4_XL.safetensors",
    cache_dir="/content/model_cache"
)
print(f"✅ 模型已下载: {model_path}")

print("⏳ 加载 ControlNet...")
cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)

print("⏳ 加载国风4...")
pipe = StableDiffusionXLControlNetPipeline.from_single_file(
    model_path, controlnet=cn, torch_dtype=torch.float16, use_safetensors=True
)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()
gc.collect(); torch.cuda.empty_cache()
print("✅ 加载完成!")

In [ ]:
# @title 4. 提示词（强力排除女性 + 强化男性历史特征）
def mkp(c):
    n = c["name"]; t = c["type"]; d = c.get("dynasty","")
    tags = ", ".join(c.get("tags",[])[:4]) if isinstance(c.get("tags"), list) else ""
    b = "masterpiece, best quality, highly detailed, 2d illustration, Chinese historical art, ink painting style"
    T = {
        "person": f"{b}, 1boy, solo, {n}, {d} dynasty Chinese historical figure, {tags}, masculine mature man, short beard, thick eyebrows, stern face, wearing traditional Chinese imperial robe or armor, ancient Chinese palace background",
        "event": f"{b}, {n}, {d} dynasty Chinese historical battle scene, {tags}, ancient armies, warriors in armor, dramatic, historical painting",
        "place": f"{b}, scenery, {n}, {d} dynasty ancient Chinese landmark, {tags}, grand architecture, palaces, city walls, no humans, Chinese ink landscape",
        "weapon": f"{b}, {n}, {d} dynasty ancient Chinese weapon, {tags}, ornate metal, still life, no humans, dark silk background",
        "classic": f"{b}, {n}, {d} dynasty ancient Chinese book, {tags}, bamboo slips, scrolls, ink brush, no humans, scholarly still life",
        "book": f"{b}, {n}, {d} dynasty ancient Chinese book, {tags}, bamboo slips, scrolls, ink brush, no humans, scholarly still life",
        "dynasty": f"{b}, {n}, {d} dynasty Chinese imperial emblem, {tags}, dragon, jade seal, palace, Great Wall, majestic",
    }
    return T.get(t, T["person"])

NEG = ("1girl, female, woman, girl, lady, feminine, girly, makeup, lipstick, long eyelashes, "
       "bishounen, pretty boy, androgynous, soft face, young boy, "
       "3d, realistic, photo, photograph, photorealistic, "
       "western, european, modern, gun, "
       "nsfw, nude, text, watermark, signature, "
       "lowres, worst quality, ugly, deformed, blurry, bad anatomy, extra fingers")
print("✅")

In [ ]:
# @title 5. 🚀 生成10张
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path("/content/drive/MyDrive")
OUT = DRIVE / "card_guofeng4_test"
OUT.mkdir(exist_ok=True)

for i, c in enumerate(cards):
    cid, name, ctype = c["card_id"], c["name"], c["type"]
    out = OUT / f"{cid}.png"
    if out.exists():
        print(f"[{i+1}/10] {name} ✓"); continue
    
    print(f"\n[{i+1}/10] {name} ({ctype})")
    try:
        ref = Image.new("RGB",(832,1216),(45,40,35))
        draw = ImageDraw.Draw(ref)
        if ctype in ("person","event"):
            draw.ellipse((340,150,492,310), outline=(120,100,80), width=5)
            draw.rectangle((312,310,520,750), outline=(120,100,80), width=5)
        elif ctype == "place":
            draw.rectangle((80,380,380,950), outline=(120,100,80), width=5)
            draw.polygon([(60,380),(230,160),(400,380)], outline=(120,100,80), width=5)
        
        edge = canny(ref, low_threshold=50, high_threshold=150)
        r = pipe(prompt=mkp(c), negative_prompt=NEG, image=edge,
                 controlnet_conditioning_scale=0.85,
                 num_inference_steps=30, guidance_scale=6.5,
                 width=832, height=1216).images[0]
        r.save(out)
        print(f"  ✅")
        gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"  ❌ {str(e)[:80]}")

done = len(list(OUT.glob("*.png")))
print(f"\n🎉 {done}/10 → Drive/card_guofeng4_test/")